# 11 — Gate 3: the manifest freeze (spec v0.12 §7, §9)

**The pre-registration moment.** Derives all 14 formulations' (w, t) vectors and
intended influence profiles, and freezes them to **`spec/manifest.csv`** +
**`spec/manifest_freeze.sha256`** (both git-tracked). Post-freeze changes require a new
manifest version + changelog entry; frozen formulations are never edited.

Roster ('formulation' = one (scenario, climate, regime) design point; 'cell' stays reserved for raster pixels): {S0…S5} × {ssp585, ssp245} + 2 crossed formulations @ ssp585 = **14**.
- **ssp245 formulations** re-derive weights with macrorefugia's swing from the 245 realization
  (constant-intended-influence, §3.2): the climate axis varies the DATA, never the values.
- **S5 (intactness-forward)** = S0 weights + `human_modification` ×10 — the Claim-B
  demonstration formulation (audit predicts near-null response; solving it makes R3 a measurement).
- **Crossed formulations** (`s1x`/`s3x`) = S1/S3 shares with the carbon regime flipped alone
  (m_soc t=0.552) — one axis at a time for E3.
- **Reference formulation** points its kbest/twin at the Gate-2 artifacts (the HiGHS twin there is
  exact and 100% integral — more authoritative than a Gurobi LP).

Zero solves. Kernel `y2y-geo`.

In [1]:
# ---- bootstrap -----------------------------------------------------------------------------
import hashlib, importlib, json, pathlib, sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc
for _m in (config, lc):
    importlib.reload(_m)

SPEC = ROOT / "analyses" / "y2y" / "spec"
AUDIT_OBJ = ROOT / "analyses" / "y2y" / "audit" / "audit_objects"
CONSTS = json.loads((AUDIT_OBJ / "audit_constants.json").read_text())
SC = json.loads((SPEC / "scenarios_v2.json").read_text())
VERDICT_RULE = "v2_8db80fed1c702638"
T_S4 = 0.552

REALIZATION = {"ssp585_2071_2100": None,   # the canonical stack layer IS the 585 realization
               "ssp245_2071_2100": config.REALIZATIONS_DIR / "macrorefugia_245_2071_2100.tif"}
sha245 = hashlib.sha256(REALIZATION["ssp245_2071_2100"].read_bytes()).hexdigest()
print(f"scenarios_v2 {SC['_meta']['spec_version']} | audit {CONSTS['created_utc']} | "
      f"verdict rule {VERDICT_RULE}")

scenarios_v2 v0.11 | audit 2026-08-26T22:32:08.502910+00:00 | verdict rule v2_8db80fed1c702638


In [2]:
# ---- derive all 14 formulations -------------------------------------------------------------------
# Scenario recipes: (block_shares, within_block, targets, extra_multipliers, regime)
# JSON stores shares at finite precision (and one historical rewrite truncated to 4
# digits) -- re-normalize on load to recover the exact intended fractions (0.25s, 1/6s),
# with a guard that catches real corruption rather than serialization residue.
def _norm(d, what):
    tot = sum(d.values())
    assert abs(tot - 1.0) < 5e-3, f"{what} sums to {tot:.6f} -- corrupted, not rounding"
    return {k: v / tot for k, v in d.items()}

S0 = SC["S0_balanced"]
recipes = {}
for name in ("S0_balanced", "S1_core_habitat", "S2_connectivity", "S3_biodiversity", "S4_carbon"):
    s = SC[name]
    recipes[name.split("_")[0].lower()] = dict(
        scenario_name=name, shares=_norm(s["block_shares"], f"{name} shares"),
        within={b: _norm(m, f"{name}/{b}") for b, m in s["within_block"].items()},
        targets=s["targets"], extra={}, regime="theta3_places" if name == "S4_carbon" else "theta5_amount")
recipes["s5"] = dict(scenario_name="S5_intactness", shares=_norm(S0["block_shares"], "S5 shares"),
                     within={b: _norm(m, f"S5/{b}") for b, m in S0["within_block"].items()}, targets=S0["targets"],
                     extra={"human_modification": 10.0}, regime="theta5_amount")
recipes["s1x"] = dict(scenario_name="S1xCarbonRegime", shares=_norm(SC["S1_core_habitat"]["block_shares"], "s1x shares"),
                      within={b: _norm(m, f"s1x/{b}") for b, m in S0["within_block"].items()}, targets={"irrecoverable_carbon_m_soc": T_S4},
                      extra={}, regime="theta3_places")
recipes["s3x"] = dict(scenario_name="S3xCarbonRegime", shares=_norm(SC["S3_biodiversity"]["block_shares"], "s3x shares"),
                      within={b: _norm(m, f"s3x/{b}") for b, m in S0["within_block"].items()}, targets={"irrecoverable_carbon_m_soc": T_S4},
                      extra={}, regime="theta3_places")

def derive(recipe, climate):
    lp = None
    if REALIZATION[climate] is not None:
        lp = {"climate_type_macrorefugia": REALIZATION[climate]}
    d = lc.scenario_weights(recipe["shares"], within_block=recipe["within"],
                            targets=recipe["targets"], layer_paths=lp)
    w = {r.feature: round(r.w, 6) for r in d.itertuples()}
    w.update(recipe["extra"])                       # S5's gHM x10 rides on top, disclosed
    prof = {r.feature: round(r.intended_share, 6) for r in d.itertuples()}
    return w, prof

forms = []
order = ["s0", "s1", "s2", "s3", "s4", "s5"]
for climate in ("ssp585_2071_2100", "ssp245_2071_2100"):
    for sid in order:
        r = recipes[sid]
        w, prof = derive(r, climate)
        forms.append(dict(sid=sid, climate=climate, r=r, w=w, prof=prof))
for sid in ("s1x", "s3x"):                          # crossed formulations @ ssp585 only
    r = recipes[sid]
    w, prof = derive(r, "ssp585_2071_2100")
    forms.append(dict(sid=sid, climate="ssp585_2071_2100", r=r, w=w, prof=prof))
assert len(forms) == 14
for c in forms:
    c["formulation_id"] = f"{c['sid']}_{c['climate'].split('_')[0]}_{c['r']['regime'].split('_')[0]}"
print(pd.DataFrame([{ "formulation_id": c["formulation_id"], **{k.split("_")[-1][:12]: v for k, v in c["w"].items()}}
                    for c in forms]).round(3).to_string(index=False))

   formulation_id  macrorefugia  connectivity  corridors   soc  biomass  birds  mammals  modification
 s0_ssp585_theta5         1.460         0.669      1.171 0.465    0.199  1.329    1.708           NaN
 s1_ssp585_theta5         3.091         0.472      0.827 0.328    0.140  0.937    1.205           NaN
 s2_ssp585_theta5         0.957         1.316      2.303 0.304    0.130  0.871    1.119           NaN
 s3_ssp585_theta5         0.782         0.358      0.627 0.249    0.106  2.134    2.743           NaN
 s4_ssp585_theta3         1.229         0.563      0.986 1.166    0.501  1.118    1.437           NaN
 s5_ssp585_theta5         1.460         0.669      1.171 0.465    0.199  1.329    1.708          10.0
 s0_ssp245_theta5         1.313         0.687      1.203 0.477    0.204  1.364    1.753           NaN
 s1_ssp245_theta5         2.864         0.500      0.874 0.347    0.148  0.992    1.275           NaN
 s2_ssp245_theta5         0.853         1.339      2.343 0.310    0.132  0.886    

In [3]:
# ---- write spec/manifest.csv + freeze hash (THE pre-registration artifact) -----------------
REF_FORMULATION = "s0_ssp585_theta5"
rows = []
now = datetime.now(timezone.utc).isoformat()
for c in forms:
    hashes = dict(CONSTS["layer_sha256"])
    if REALIZATION[c["climate"]] is not None:
        hashes["climate_type_macrorefugia"] = sha245     # the layer this formulation ACTUALLY solves on
    rows.append(dict(
        formulation_id=c["formulation_id"], scenario_id=c["sid"], scenario_name=c["r"]["scenario_name"],
        climate_level=c["climate"], carbon_regime=c["r"]["regime"],
        budget_pct=config.BUDGET_PCT,
        weight_vector=json.dumps(c["w"]), target_vector=json.dumps(c["r"]["targets"]),
        influence_profile_intended=json.dumps(c["prof"]),
        k_requested=50, band_gap_g=0.05, opt_gap=1e-4, numeric_focus=2,
        dust_rule_version="1e-9/2026-08-26", estimator="mga_maxham_v1",
        verdict_rule=VERDICT_RULE, solver="gurobi",
        seed_policy="deterministic (MGA warm-start chain; no RNG)",
        input_layer_hashes=json.dumps(hashes),
        kbest_ref="output_data/iter9_y2y_s0_pool" if c["formulation_id"] == REF_FORMULATION else "",
        twin_ref="output_data/iter9_y2y_s0_lp" if c["formulation_id"] == REF_FORMULATION else "",
        created_utc=now, frozen=True))
M = pd.DataFrame(rows)
assert M.formulation_id.is_unique and len(M) == 14
for ref in ("kbest_ref", "twin_ref"):               # pointers must resolve
    for p in M.loc[M[ref] != "", ref]:
        assert (ROOT / p / "run_summary.json").exists(), f"{ref} does not resolve: {p}"
out = SPEC / "manifest.csv"
M.to_csv(out, index=False)
digest = hashlib.sha256(out.read_bytes()).hexdigest()
(SPEC / "manifest_freeze.sha256").write_text(f"{digest}  manifest.csv\n")
print(f"FROZEN: {out.relative_to(ROOT)} (14 formulations) | sha256 {digest[:16]}...")
print("commit spec/manifest.csv + spec/manifest_freeze.sha256 -- that commit IS the pre-registration")

FROZEN: analyses/y2y/spec/manifest.csv (14 formulations) | sha256 d1723c82864f48c1...
commit spec/manifest.csv + spec/manifest_freeze.sha256 -- that commit IS the pre-registration


## Next

Commit the freeze, then `12_gate4_ensemble.ipynb` (R) — its first executed cell is a
NO-SOLVE dry plan; eyeball the worklist before letting the loop run (~10 h serial,
resumable, live internet).